In [0]:
%run "/Workspace/utilities"

In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")

# COMMAND ----------

# Use config from utilities
logger = DataLogger("customer_segmentation")

metrics_path = config.get_gold_path("customer_metrics")
segments_path = config.get_gold_path("customer_segments")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Customer Metrics

# COMMAND ----------

df_metrics = spark.read.format("delta").load(metrics_path)
logger.info(f"Loaded {df_metrics.count()} customer metrics")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Define Segmentation Logic

# COMMAND ----------

from pyspark.sql.functions import *

# Create customer segments based on RFM scores
df_segmented = df_metrics.withColumn(
    "customer_segment",
    when(
        (col("recency_score") >= 4) & 
        (col("frequency_score") >= 4) & 
        (col("monetary_score") >= 4),
        "Champions"
    )
    .when(
        (col("frequency_score") >= 4) & 
        (col("monetary_score") >= 4),
        "Loyal Customers"
    )
    .when(
        (col("recency_score") <= 2) & 
        (col("frequency_score") >= 3) & 
        (col("monetary_score") >= 3),
        "At Risk"
    )
    .when(
        (col("recency_score") >= 4) & 
        (col("frequency_score") <= 2),
        "New Customers"
    )
    .when(
        (col("recency_score") <= 2) & 
        (col("frequency_score") <= 2),
        "Lost Customers"
    )
    .when(
        (col("monetary_score") >= 4),
        "Big Spenders"
    )
    .when(
        (col("frequency_score") >= 3),
        "Potential Loyalists"
    )
    .otherwise("Others")
)

# Add segment-specific recommendations
df_segmented = df_segmented.withColumn(
    "marketing_action",
    when(col("customer_segment") == "Champions", 
         "VIP rewards, early access to new products")
    .when(col("customer_segment") == "Loyal Customers", 
          "Upsell, cross-sell opportunities")
    .when(col("customer_segment") == "At Risk", 
          "Win-back campaigns, special offers")
    .when(col("customer_segment") == "New Customers", 
          "Onboarding series, product education")
    .when(col("customer_segment") == "Lost Customers", 
          "Re-engagement campaigns, surveys")
    .when(col("customer_segment") == "Big Spenders", 
          "Premium products, exclusive deals")
    .otherwise("General campaigns")
)

logger.info("Customer segmentation completed")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Add Business Value Indicators

# COMMAND ----------

# Assign priority scores
df_segmented = df_segmented.withColumn(
    "priority_score",
    when(col("customer_segment").isin(["Champions", "Loyal Customers"]), 5)
    .when(col("customer_segment").isin(["At Risk", "Big Spenders"]), 4)
    .when(col("customer_segment") == "Potential Loyalists", 3)
    .when(col("customer_segment") == "New Customers", 2)
    .otherwise(1)
)

# Calculate segment value
df_segmented = df_segmented.withColumn(
    "segment_lifetime_value_category",
    when(col("customer_segment").isin(["Champions", "Loyal Customers", "Big Spenders"]), "High")
    .when(col("customer_segment").isin(["At Risk", "Potential Loyalists"]), "Medium")
    .otherwise("Low")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Gold Layer

# COMMAND ----------

try:
    df_segmented.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("customer_segment") \
        .save(segments_path)
    
    logger.info(f"✅ Successfully wrote {df_segmented.count()} customer segments to Gold layer")
    
except Exception as e:
    logger.error("Failed to write to Gold layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Segment Analysis

# COMMAND ----------

print("=== Customer Segment Distribution ===\n")

segment_summary = df_segmented.groupBy("customer_segment").agg(
    count("*").alias("customer_count"),
    avg("monetary").alias("avg_revenue"),
    sum("monetary").alias("total_revenue"),
    avg("frequency").alias("avg_orders"),
    avg("recency_days").alias("avg_recency")
).orderBy("total_revenue", ascending=False)

display(segment_summary)

# COMMAND ----------

# Segment performance by country
print("\n=== Top Segments by Country ===")

country_segments = df_segmented.groupBy("country", "customer_segment").agg(
    count("*").alias("customer_count")
).orderBy("customer_count", ascending=False).limit(20)

display(country_segments)

# COMMAND ----------

logger.info("Customer segmentation completed successfully")